In [ ]:
import numpy as np
from datetime import datetime, timedelta

# READ FILE

file = open("GroupDNA_privacy_variant.txt", "r", encoding="utf-8")
lines = file.readlines()
file.close()

messages = []

for line in lines:

    line = line.strip()

    if " - " not in line:
        continue

    a = line.split(" - ", 1)

    if ": " not in a[1]:
        continue

    b = a[1].split(": ", 1)

    try:
        date_time = datetime.strptime(a[0], "%d/%m/%y, %H:%M")

    except:
        continue

    name = b[0]
    text = b[1]

    if text == "<Media omitted>":
        continue

    if "This message was deleted" in text:
        continue

    messages.append([date_time, name, text])


#  PARTICIPANTS

people = []
count = {}

for msg in messages:

    name = msg[1]

    if name not in people:
        people.append(name)
        count[name] = 0

    count[name] += 1

people.sort(key=lambda x: count[x],reverse=True)


# DATE

start = messages[0][0].date()
end = messages[-1][0].date()

total_days = (end - start).days + 1


#  BUSIEST DAY

day_count = {}

for msg in messages:
  day = msg[0].date()

  if day not in day_count:
    day_count[day] = 0

    day_count[day] += 1

busy_day = max(day_count,key=day_count.get)


#  BUSIEST HOUR

hour_count = {}

for msg in messages:
  hour = msg[0].hour

  if hour not in hour_count:
    hour_count[hour] = 0

    hour_count[hour] += 1

busy_hour = max(hour_count,key=hour_count.get)


#  NUMPY ACTIVITY

activity = np.zeros((len(people), 24),dtype=int)

for msg in messages:
  person = people.index(msg[1])
  hour = msg[0].hour

  activity[person][hour] += 1


#  TOP WORDS

stop_words = [
    "the", "and", "you", "that", "this",
    "was", "are", "for", "with", "have",
    "has", "had", "from", "your", "they",
    "them", "but", "not", "what", "how",
    "why", "who", "can", "will", "is",
    "am", "to", "of", "in", "on", "a"
]

words_count = {}

for msg in messages:

    words = msg[2].lower().split()

    for word in words:

        word = word.strip(".,!?;:\"'()[]{}")

        if len(word) <= 2:
            continue

        if word in stop_words:
            continue

        if word not in words_count:
            words_count[word] = 0

        words_count[word] += 1


top_words = sorted(words_count.items(),key=lambda x: x[1],reverse=True)


#  RESPONSE TIME

response_total = {}
response_count = {}

for i in range(1, len(messages)):

    previous = messages[i - 1]
    current = messages[i]

    if previous[1] == current[1]:
        continue

    minutes = (current[0] - previous[0]).total_seconds() / 60

    if minutes < 0 or minutes > 1440:
        continue

    name = current[1]

    if name not in response_total:
        response_total[name] = 0
        response_count[name] = 0

    response_total[name] += minutes
    response_count[name] += 1


average_response = {}

for person in people:

    if person in response_count:
      average_response[person] = (response_total[person]/ response_count[person])

    else:
      average_response[person] = 0


# SILENT DAYS

dates = []
day = start

while day <= end:

    dates.append(day)

    day = day + timedelta(days=1)


silent = {}

for person in people:

    active = set()

    for msg in messages:

        if msg[1] == person:
            active.add(msg[0].date())

    longest = 0
    current = 0

    for day in dates:

        if day not in active:
          current += 1

        if current > longest:
          longest = current

        else:
          current = 0

    silent[person] = longest


#  ARCHETYPES

roles = [
    "THE SPAMMER",
    "THE GROUP MOM",
    "THE NIGHT OWL",
    "THE STORYTELLER",
    "THE DRAMA QUEEN",
    "THE GHOST",
    "THE COMEDIAN",
    "THE QUESTION MASTER"
]

score = {}

for person in people:

    score[person] = {}

    for role in roles:
        score[person][role] = 0


# Spammer

for person in people:
    burst = 0
    total_burst = 0
    bursts = 0

    for msg in messages:
        if msg[1] == person:
            burst += 1

        else:
            if burst > 0:
                total_burst += burst
                bursts += 1
            burst = 0

    if burst > 0:
        total_burst += burst
        bursts += 1

    if bursts > 0:
        score[person]["THE SPAMMER"] = (total_burst / bursts)


# Group Mom

care_words = ["okay", "safe", "eat", "sleep","take care", "please", "reminder","don't forget"]

for person in people:
    value = 0

    for msg in messages:
        if msg[1] == person:
            text = msg[2].lower()

            for word in care_words:
                if word in text:
                    value += 1

    score[person]["THE GROUP MOM"] = value


# Night Owl

for person in people:
    night = 0

    for msg in messages:
        if msg[1] == person:
            hour = msg[0].hour

            if hour >= 23 or hour <= 4:
                night += 1

    score[person]["THE NIGHT OWL"] = (night / count[person])


# Storyteller

for person in people:
    total_words = 0

    for msg in messages:
        if msg[1] == person:
            total_words += len(msg[2].split())

    score[person]["THE STORYTELLER"] = (total_words / count[person])


# Drama Queen

for person in people:
    value = 0

    for msg in messages:
        if msg[1] == person:
            text = msg[2]

            if text.count("!") >= 2:
                value += 1

            if len(text) > 3 and text.isupper():
                value += 1

    score[person]["THE DRAMA QUEEN"] = value


# Ghost

for person in people:
    score[person]["THE GHOST"] = (silent[person] / total_days)


# Comedian

funny_words = ["haha", "hahaha", "lol","lmao", "rofl"]

for person in people:
    value = 0

    for msg in messages:
        if msg[1] == person:
            text = msg[2].lower()

            for word in funny_words:
                if word in text:
                    value += 1

    score[person]["THE COMEDIAN"] = (value / count[person])


# Question Master

for person in people:
    questions = 0

    for msg in messages:
        if msg[1] == person:

            if msg[2].strip().endswith("?"):
                questions += 1

    score[person]["THE QUESTION MASTER"] = (questions / count[person])


# One role for each person

archetype = {}

for person in people:
    best = max(score[person],key=score[person].get)

    archetype[person] = best


# FINAL REPORT


print()

print("                      # GROUPDNA #")
print()


print("\n//GROUP OVERVIEW//\n")


print("Period:",start.strftime("%d/%m/%Y"),"to",end.strftime("%d/%m/%Y"))

print("Total Messages:", len(messages))
print("Total Participants:", len(people))

print("\nMessages per participant:\n")

for person in people:

    print(person,":",count[person])


print("\n//ACTIVITY//\n")


print("Busiest Day:",busy_day.strftime("%d/%m/%Y"),"| Messages:",day_count[busy_day])


print(
    "Busiest Hour:",
    str(busy_hour).zfill(2) + ":00 - " +
    str((busy_hour + 1) % 24).zfill(2) + ":00",
    "| Messages:",
    hour_count[busy_hour]
)

print("\n//ACTIVITY HEATMAP//\n")


print("Name       00   03   06   09   12   15   18   21")


for i in range(len(people)):

    print(people[i].ljust(10), end="")

    for h in range(0, 24, 3):

        value = sum(activity[i][h:h+3])

        if value == 0:
            symbol = "."
        elif value < 10:
            symbol = "░"
        elif value < 25:
            symbol = "▒"
        elif value < 40:
            symbol = "▓"
        else:
            symbol = "█"

        print(symbol.center(5), end="")

    print()
    print()


print("\n//TOP 10 WORDS//\n")


for word, number in top_words[:10]:
    print(word.ljust(12),":",number)


print("\n//RESPONSE PATTERNS//\n")

fastest = min(people,key=average_response.get)

slowest = max(people,key=average_response.get)

print("Fastest Responder:",fastest,"|",round(average_response[fastest], 2),"minutes")

print("Slowest Responder:",slowest,"|",round(average_response[slowest], 2),"minutes")


print("\n//LONGEST SILENT STREAK//\n")


for person in people:
    print(person.ljust(10),":",silent[person],"days")


print("\n//PERSONALITY ARCHETYPES//\n")


for person in people:
    print(person.ljust(10),"→",archetype[person])


print("\n//PARTICIPANT PROFILE//\n")


print("Name       Messages   Contribution   Response   Peak")

for person in people:
    percentage = (count[person] / len(messages)) * 100

    row = people.index(person)

    peak = np.argmax(activity[row])

    print(person.ljust(10),str(count[person]).ljust(10),str(round(percentage, 2)).ljust(14),str(round(average_response[person], 2)).ljust(10),str(peak).zfill(2) + ":00")


print()

print("                  # GROUPDNA COMPLETE #")



                      # GROUPDNA #


//GROUP OVERVIEW//

Period: 01/04/2024 to 30/05/2024
Total Messages: 3127
Total Participants: 6

Messages per participant:

Omkar : 940
Ananya : 712
Meera : 624
Aditya : 484
Kavya : 345
Yash : 22

//ACTIVITY//

Busiest Day: 01/04/2024 | Messages: 1
Busiest Hour: 01:00 - 02:00 | Messages: 1

//ACTIVITY HEATMAP//

Name       00   03   06   09   12   15   18   21
Omkar       ▓    █    █    █    █    █    █    █  

Ananya      .    .    █    █    █    █    █    █  

Meera       .    ▒    █    █    █    █    █    █  

Aditya      █    █    .    .    ▒    ▓    ▓    █  

Kavya       .    .    ▒    █    █    █    █    ▓  

Yash        .    .    ░    ░    ░    ░    ░    ░  


//TOP 10 WORDS//

guys         : 318
today        : 295
about        : 274
hai          : 268
his          : 217
just         : 208
which        : 202
plan         : 187
everyone     : 187
telling      : 179

//RESPONSE PATTERNS//

Fastest Responder: Yash | 34.9 minutes
Slowest Respond